### Purpose
This notebook does the following:
1. Reads a file from the landing zone.
2. Adds control columns such as processing date and file metadata.
3. Moves the data to a Delta table as-is by always appending new information.
4. Checks and maintains the Delta table.

In [ ]:
# Get notebook parameters from the Azure Data Factory pipeline
dbutils.widgets.text("_pipeline_run_id","0478ce36-b895-48a0-8a08-1b10430247ca")
dbutils.widgets.text("_filename","nybabynames.csv")
dbutils.widgets.text("_processing_date","21-05-2024 18:39:52")
dbutils.widgets.text("_account_name","datalakexxxxx")
_pipeline_run_id = dbutils.widgets.get("_pipeline_run_id")
_filename = dbutils.widgets.get("_filename")
_processing_date = dbutils.widgets.get("_processing_date")
accountName = dbutils.widgets.get("_account_name")
print(_processing_date)
print(_pipeline_run_id)
print(accountName)
print(_filename)

In [ ]:
# Unity Catalog external locations handle ADLS authentication via the Access Connector managed identity.
# No account key or secret scope is needed — just the storage account name (passed as a notebook parameter).

In [ ]:
# Define the location
landingSource = f'abfss://landing@{accountName}.dfs.core.windows.net/{_filename}'
bronzeTarget = f'abfss://bronze@{accountName}.dfs.core.windows.net/nybabynames'

# Bronze Delta Table
table_name = "bronze.new_york_baby_names"

In [ ]:
# Read csv file data from Data Lake with explicit schema (avoids double-read from inferSchema)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import json

landing_schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("county", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("name_count", IntegerType(), True),
])

gridDataDf = spark.read.schema(landing_schema).csv(path=landingSource, header=True)

if gridDataDf.limit(1).count() == 0:
    dbutils.notebook.exit(json.dumps({"status": "no_data", "table": table_name, "message": "No rows found in landing file"}))

gridDataDf.printSchema()



In [ ]:
from pyspark.sql.functions import lit, col, input_file_name
from datetime import datetime

# Add audit columns to the DataFrame

# 1. Processing timestamp from pipeline input
# 2. Pipeline run ID from ADF
# 3. Input file name for traceability
# 4. Input file modification date from file metadata
gridDataDf = gridDataDf.withColumn("_processing_date", lit(datetime.strptime(_processing_date, '%d-%m-%Y %H:%M:%S'))) \
                       .withColumn("_pipeline_run_id", lit(_pipeline_run_id)) \
                       .withColumn("_input_filename", input_file_name()) \
                       .withColumn("_input_file_modification_date", col("_metadata.file_modification_time"))

gridDataDf.printSchema()


In [ ]:
from delta.tables import DeltaTable

# Check whether the bronze path already contains a Delta table
if DeltaTable.isDeltaTable(spark, bronzeTarget):

    # If yes, append data to the existing Delta table
    gridDataDf.write.mode("append").format("delta").save(bronzeTarget)
else:

    # If no, save the file to bronze
    gridDataDf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronzeTarget)

In [ ]:
# Create the schema and table, if required

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {table_name} USING delta LOCATION '{bronzeTarget}'")

# Note: spark.sql is used here to inject the bronze path with an f-string.

In [ ]:
%sql
-- This is not necessary from a pipeline perspective; it involves checking table information as a learning experience.

DESCRIBE EXTENDED bronze.new_york_baby_names

-- Location: stored in the storage account
-- Provider (format): Delta

In [ ]:
%sql
-- This is not necessary from a pipeline perspective; it involves showing the transaction log on the delta version as a learning experience.

SELECT version, operationMetrics, operationMetrics.numOutputRows, operationMetrics.numTargetRowsInserted, operationMetrics.numTargetRowsUpdated, operationMetrics.numTargetRowsDeleted
FROM (DESCRIBE HISTORY bronze.new_york_baby_names)


In [ ]:
%sql

-- Check your result for testing. Do not do this in production!
-- SELECT *
-- FROM bronze.new_york_baby_names



In [ ]:
# Maintenance for Delta table

# To optimize Delta table performance, we run two commands:
# 1. optimize(): compacts small files.
# 2. vacuum(): removes old files. This reduces overhead but limits time travel.


# Databricks recommends frequently running OPTIMIZE to compact small files.
# This operation does not remove old files. To remove them, run VACUUM (https://learn.microsoft.com/en-us/azure/databricks/delta/vacuum).
# https://learn.microsoft.com/en-us/azure/databricks/delta/best-practices#--compact-files

# In Azure, predictive optimization can be used (https://learn.microsoft.com/en-us/azure/databricks/optimizations/predictive-optimization#what-operations-does-predictive-optimization-run).
# It has prerequisites, such as Premium plan and managed tables (https://learn.microsoft.com/en-us/azure/databricks/optimizations/predictive-optimization#prerequisites-for-predictive-optimization).

gridDataDelta = DeltaTable.forName(spark, table_name)

# In this example, we run optimize and vacuum every 30 days
if gridDataDelta.history(30).filter("operation = 'VACUUM START'").count() == 0:
    gridDataDelta.optimize()
    gridDataDelta.vacuum() # default = 7 days

import json
dbutils.notebook.exit(json.dumps({"status": "success", "table": table_name}))